### Week 13 notebook - Naive Bayes (Gaussian), Mod C Data Science Capstone Fall 2026
### By Shuqin Ouyang

# Overall notebook structure:

Breadth week, so only the main topic, Gaussian Naive Bayes, run on both datasets:
1. Why only the Gaussian variant (Bernoulli and Multinomial are explained but not run)
2. A check of the two Naive Bayes assumptions on each dataset (independence and bell shape)
3. Gaussian Naive Bayes results: F1, precision, recall and overconfidence
4. Conclusion and Summary

The setup is the same as Weeks 5 to 11: the 80/20 split. No scaling is needed, because Gaussian Naive Bayes fits a mean and a spread for each feature by itself.

# Student dataset analysis starts here:

Naive Bayes: what it is and which variant fits my data.

Naive Bayes is a generative classifier. For each class it asks how likely the observed features are, multiplies that by how common the class is (the prior), and compares the classes (Bayes' rule). It multiplies one likelihood per feature, as if the features were independent, and that is the "naive" part. The variants differ in how the likelihood of one feature is calculated:
- Gaussian: for continuous numbers. For each class it learns the feature's average and spread, and uses a bell curve.
- Bernoulli: for yes/no features, such as a word being present or not. For each class it learns how often the feature is present.
- Multinomial: for counts, such as how many times a word appears. For each class it learns how often each feature occurs.

My features are continuous numbers (financial ratios, amounts), so only Gaussian fits directly. I do not run Bernoulli, because I have no yes/no features, and turning numbers into above or below the median would throw away information. I do not run Multinomial, because I have no counts. The closest are the credit card repayment status columns, but they include negative values that Multinomial cannot take. Gaussian assumes each feature is bell-shaped within each class and that the features are independent. Both are probably partly broken in my data, so this week checks both assumptions and what they cost.

In [1]:
# Imports
import pandas as pd
import numpy as np
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import train_test_split
from sklearn.metrics import f1_score, precision_score, recall_score

Dataset 1 background quick overview:

The dataset 1 used is the Taiwanese Bankruptcy Prediction dataset from UCI, link as https://archive.ics.uci.edu/dataset/572/taiwanese+bankruptcy+prediction.
It contains company financial indicators and a binary outcome for bankruptcy.

In [2]:
# install uci package so can load the data using package
# install once in terminal or notebook

!pip install ucimlrepo

from ucimlrepo import fetch_ucirepo

# load dataset from UCI repository using dataset id
data_bankruptcy = fetch_ucirepo(id=572)

df_bankruptcy = pd.concat(
    [data_bankruptcy.data.features.copy(),
     data_bankruptcy.data.targets.copy()],
    axis=1
)

# remove leading/trailing spaces from column names once, here
df_bankruptcy.columns = df_bankruptcy.columns.str.strip()

df_bankruptcy.head()

'pip' is not recognized as an internal or external command,
operable program or batch file.


,ROA(C) before interest and depreciation before interest,ROA(A) before interest and % after tax,ROA(B) before interest and depreciation after tax,Operating Gross Margin,Realized Sales Gross Margin,Operating Profit Rate,Pre-tax net Interest Rate,After-tax net Interest Rate,Non-industry income and expenditure/revenue,Continuous interest rate (after tax),...,Total assets to GNP price,No-credit Interval,Gross Profit to Sales,Net Income to Stockholder's Equity,Liability to Equity,Degree of Financial Leverage (DFL),Interest Coverage Ratio (Interest expense to EBIT),Net Income Flag,Equity to Liability,Bankrupt?
0,0.370594,0.424389,0.405750,0.601457,0.601457,0.998969,0.796887,0.808809,0.302646,0.780985,...,0.009219,0.622879,0.601453,0.827890,0.290202,0.026601,0.564050,1,0.016469,1
1,0.464291,0.538214,0.516730,0.610235,0.610235,0.998946,0.797380,0.809301,0.303556,0.781506,...,0.008323,0.623652,0.610237,0.839969,0.283846,0.264577,0.570175,1,0.020794,1
2,0.426071,0.499019,0.472295,0.601450,0.601364,0.998857,0.796403,0.808388,0.302035,0.780284,...,0.040003,0.623841,0.601449,0.836774,0.290189,0.026555,0.563706,1,0.016474,1
3,0.399844,0.451265,0.457733,0.583541,0.583541,0.998700,0.796967,0.808966,0.303350,0.781241,...,0.003252,0.622929,0.583538,0.834697,0.281721,0.026697,0.564663,1,0.023982,1
4,0.465022,0.538432,0.522298,0.598783,0.598783,0.998973,0.797366,0.809304,0.303475,0.781550,...,0.003878,0.623521,0.598782,0.839973,0.278514,0.024752,0.575617,1,0.035490,1


Student markdown:

Dataset 1 is the Taiwanese bankruptcy data: 6,819 companies described by 95 financial ratios, of which about 3.2% went bankrupt. All features are numeric and none are missing.

In [3]:
X_train_bankrupt, X_test_bankrupt, y_train_bankrupt, y_test_bankrupt = train_test_split(df_bankruptcy.drop(columns='Bankrupt?'), df_bankruptcy['Bankrupt?'], test_size=0.2, stratify=df_bankruptcy['Bankrupt?'], random_state=42)    # same split as Weeks 5, 6, 8 and 9

Check of the two assumptions, on the training rows. Independence: the number of feature pairs with a correlation above 0.9 (like "ice" and "cream", features that move together, so their evidence would be counted twice). Bell shape: the number of features with strong skew (absolute skewness above 1), a quick sign that a feature is not bell-shaped.

In [4]:
corr_bankrupt = X_train_bankrupt.corr().abs().values
print("feature pairs with correlation above 0.9:", (np.triu(corr_bankrupt, k=1) > 0.9).sum(), "of", len(corr_bankrupt) * (len(corr_bankrupt) - 1) // 2)
print("features with strong skew (absolute skewness above 1):", (X_train_bankrupt.skew().abs() > 1).sum(), "of", X_train_bankrupt.shape[1])

feature pairs with correlation above 0.9: 36 of 4465
features with strong skew (absolute skewness above 1): 82 of 95


Student markdown:

For bankruptcy, the assumptions are clearly broken. 82 of 95 features have strong skew, so most are not bell-shaped. Only 36 of 4,465 feature pairs have a correlation above 0.9, a small share of pairs, but those features carry the same information more than once. So the bell shape is the bigger problem here, and the independence is broken for a smaller part of the data.

Gaussian Naive Bayes is fit on the training rows and judged on the unseen test rows with F1, as in Weeks 8 and 9. The baseline F1 is what you get by predicting every row as failed. The train against test F1 shows overfitting, and the last line shows overconfidence: the share of test rows where the model gives a probability above 0.99 or below 0.01.

In [5]:
nb_bankrupt = GaussianNB().fit(X_train_bankrupt, y_train_bankrupt)
pred_train_bankrupt = nb_bankrupt.predict(X_train_bankrupt)
pred_test_bankrupt = nb_bankrupt.predict(X_test_bankrupt)
proba_test_bankrupt = nb_bankrupt.predict_proba(X_test_bankrupt)[:, 1]
p_bankrupt = y_test_bankrupt.mean()
print("class prior (not failed, failed):", nb_bankrupt.class_prior_.round(3))
print("baseline F1 (predict every row as failed):", round(2 * p_bankrupt / (1 + p_bankrupt), 3))
print("train F1:", round(f1_score(y_train_bankrupt, pred_train_bankrupt, zero_division=0), 3), "| test F1:", round(f1_score(y_test_bankrupt, pred_test_bankrupt, zero_division=0), 3))
print("test precision:", round(precision_score(y_test_bankrupt, pred_test_bankrupt, zero_division=0), 3), "| test recall:", round(recall_score(y_test_bankrupt, pred_test_bankrupt), 3))
print("share of test rows with probability above 0.99 or below 0.01:", round(((proba_test_bankrupt > 0.99) | (proba_test_bankrupt < 0.01)).mean(), 3))

class prior (not failed, failed): [0.968 0.032]
baseline F1 (predict every row as failed): 0.062
train F1: 0.064 | test F1: 0.062
test precision: 0.032 | test recall: 0.955
share of test rows with probability above 0.99 or below 0.01: 0.999


Student markdown:

Gaussian Naive Bayes does not beat the baseline for bankruptcy. The test F1 is 0.062, the same as the baseline of 0.062, and the train F1 is 0.064, so there is no overfitting gap but also no skill. The precision of 0.032 equals the failure rate and the recall is 0.955, so the model flags almost every company as failed, which is what the baseline does. It is also extremely overconfident: 0.999 of the test rows get a probability above 0.99 or below 0.01. With 82 of 95 features not bell-shaped, and 95 likelihoods multiplied together, I think the probabilities are pushed to one side, but I did not test the cause directly.

Dataset 2 background quick overview:

The dataset 2 used is the Default of Credit Card Clients dataset from UCI, link as https://archive.ics.uci.edu/dataset/350/default+of+credit+card+clients.

It contains demographic information, credit limits, repayment history, bill statements, and previous payment records for Taiwanese credit card clients, along with a binary outcome indicating whether the client defaulted on the next payment.

In [6]:
# UCI dataset already decides features and targets already, but still need to look into details / verify to make sure all ok
data_credit = fetch_ucirepo(id=350)

df_credit = pd.concat(
    [data_credit.data.features.copy(),
     data_credit.data.targets.copy()],
    axis=1
)

df_credit.head()

,X1,X2,X3,X4,X5,X6,X7,X8,X9,X10,...,X15,X16,X17,X18,X19,X20,X21,X22,X23,Y
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [7]:
# columns rename for easier reading

df_credit = df_credit.rename(columns={

    "Y": "default_next_month",

    "X1": "credit_limit",
    "X2": "gender",
    "X3": "education",
    "X4": "marital_status",
    "X5": "age",

    "X6": "repay_2005_sep",
    "X7": "repay_2005_aug",
    "X8": "repay_2005_jul",
    "X9": "repay_2005_jun",
    "X10": "repay_2005_may",
    "X11": "repay_2005_apr",

    "X12": "bill_2005_sep",
    "X13": "bill_2005_aug",
    "X14": "bill_2005_jul",
    "X15": "bill_2005_jun",
    "X16": "bill_2005_may",
    "X17": "bill_2005_apr",

    "X18": "pay_2005_sep",
    "X19": "pay_2005_aug",
    "X20": "pay_2005_jul",
    "X21": "pay_2005_jun",
    "X22": "pay_2005_may",
    "X23": "pay_2005_apr"
})

df_credit.head()

,credit_limit,gender,education,marital_status,age,repay_2005_sep,repay_2005_aug,repay_2005_jul,repay_2005_jun,repay_2005_may,...,bill_2005_jun,bill_2005_may,bill_2005_apr,pay_2005_sep,pay_2005_aug,pay_2005_jul,pay_2005_jun,pay_2005_may,pay_2005_apr,default_next_month
0,20000,2,2,1,24,2,2,-1,-1,-2,...,0,0,0,0,689,0,0,0,0,1
1,120000,2,2,2,26,-1,2,0,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,90000,2,2,2,34,0,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,50000,2,2,1,37,0,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,50000,1,2,1,57,-1,0,-1,0,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


Student markdown:

Dataset 2 is the credit card default data: 30,000 clients described by 23 features (credit limit, demographics, six months of repayment status, bill amounts and payments), of which about 22% default next month. No values are missing.

In [8]:
X_train_credit, X_test_credit, y_train_credit, y_test_credit = train_test_split(df_credit.drop(columns='default_next_month'), df_credit['default_next_month'], test_size=0.2, stratify=df_credit['default_next_month'], random_state=42)    # same split as Weeks 5, 6, 8 and 9

Check of the two assumptions for credit card: the same two counts as for bankruptcy.

In [9]:
corr_credit = X_train_credit.corr().abs().values
print("feature pairs with correlation above 0.9:", (np.triu(corr_credit, k=1) > 0.9).sum(), "of", len(corr_credit) * (len(corr_credit) - 1) // 2)
print("features with strong skew (absolute skewness above 1):", (X_train_credit.skew().abs() > 1).sum(), "of", X_train_credit.shape[1])

feature pairs with correlation above 0.9: 7 of 253
features with strong skew (absolute skewness above 1): 13 of 23


Student markdown:

Credit card breaks the assumptions less. 13 of 23 features have strong skew, and 7 of 253 feature pairs have a correlation above 0.9, probably the monthly bill columns, which move together (I did not list the pairs).

Gaussian Naive Bayes for credit card: the same fit and the same printed results as for bankruptcy.

In [10]:
nb_credit = GaussianNB().fit(X_train_credit, y_train_credit)
pred_train_credit = nb_credit.predict(X_train_credit)
pred_test_credit = nb_credit.predict(X_test_credit)
proba_test_credit = nb_credit.predict_proba(X_test_credit)[:, 1]
p_credit = y_test_credit.mean()
print("class prior (not failed, failed):", nb_credit.class_prior_.round(3))
print("baseline F1 (predict every row as failed):", round(2 * p_credit / (1 + p_credit), 3))
print("train F1:", round(f1_score(y_train_credit, pred_train_credit, zero_division=0), 3), "| test F1:", round(f1_score(y_test_credit, pred_test_credit, zero_division=0), 3))
print("test precision:", round(precision_score(y_test_credit, pred_test_credit, zero_division=0), 3), "| test recall:", round(recall_score(y_test_credit, pred_test_credit), 3))
print("share of test rows with probability above 0.99 or below 0.01:", round(((proba_test_credit > 0.99) | (proba_test_credit < 0.01)).mean(), 3))

class prior (not failed, failed): [0.779 0.221]
baseline F1 (predict every row as failed): 0.362
train F1: 0.391 | test F1: 0.382
test precision: 0.25 | test recall: 0.818
share of test rows with probability above 0.99 or below 0.01: 0.086


Student markdown:

Gaussian Naive Bayes only slightly beats the baseline for credit card: the test F1 is 0.382 against 0.362, and the train F1 is 0.391, so the gap is small and there is no overfitting. The recall is high (0.818) and the precision is low (0.25, against a failure rate of 0.221), so the model flags many clients as defaulting and is right only a little more often than guessing. It is much less overconfident than for bankruptcy (0.086 of the test rows have an extreme probability), which fits the credit card features breaking the assumptions less.

Student markdown:

Conclusion:

**Overfitting:** there is no overfitting gap: train and test F1 are 0.064 and 0.062 for bankruptcy, and 0.391 and 0.382 for credit card. This is not a good sign here, because the model has almost no skill (the test F1 equals or barely beats the baseline). It is the opposite of Week 9, where the bankruptcy gradient boosting model had a train F1 of 1.0 against a cross-validated F1 of 0.353. Naive Bayes is too simple for this data, because its assumptions are broken. I used only the default settings and did not try other probability thresholds.

**Metrics and results:** I use test F1 against the baseline F1, with precision and recall to see what the model is doing, as in Weeks 8 and 9, plus the share of extreme probabilities to see overconfidence. For bankruptcy, the test F1 is 0.062, equal to the baseline, with precision 0.032 and recall 0.955, so it flags almost every company. For credit card, it is 0.382 against 0.362, with precision 0.25 and recall 0.818. In Week 9 (gradient boosting) the test F1 was 0.349 for bankruptcy and 0.478 for credit card, so Naive Bayes is worse for both.

**Expected vs. unexpected:**
- Expected: the two assumptions to be partly broken, so Naive Bayes would not do well. The checks confirmed it: 82 of 95 bankruptcy features and 13 of 23 credit card features are strongly skewed, and 36 and 7 feature pairs are highly correlated.
- Unexpected, bankruptcy: the result was exactly the baseline (test F1 0.062, precision 0.032, recall 0.955), and 0.999 of the test probabilities are extreme.
- Unexpected, credit card: it still slightly beats the baseline (0.382 against 0.362), with much less overconfidence (0.086).

**EDA usefulness:** the two checks were the useful part, because they showed that the assumptions were broken before I looked at the F1. Bankruptcy has a larger share of skewed features (82 of 95 against 13 of 23) and the worse result, which is consistent with the broken assumptions mattering, but with two datasets I cannot show that this is the cause. The checks did not say which features cause the problem.

**Bigger picture, research questions:**
- RQ1, which risk signals keep appearing: this notebook adds nothing, because I did not look at which features drive the Naive Bayes predictions, and the model has almost no skill, so it gives no reliable signal.
- RQ2, whether bankruptcy is harder to model: for this model, yes. Bankruptcy gets no gain over the baseline (0.062 against 0.062), while credit card gets a small gain (0.382 against 0.362).
- RQ3, whether results stay stable across model types: not really. Naive Bayes is worse than the Week 9 model for both datasets. Credit card has the higher F1 in both models (0.382 against 0.062 here, 0.478 against 0.349 in Week 9), but how well the model copes depends on how well the data fits the assumptions, and bankruptcy fits them worse.

**How I know the conclusions are true:** it is one split with default settings and the default 0.5 threshold, and the bankruptcy test set has only 44 failures, so its scores are noisy. The Week 9 numbers come from the Week 9 notebook, not this one. The cut-offs for the checks (a correlation of 0.9 and a skewness of 1) are my choices. The idea that the broken assumptions cause the bankruptcy result is not tested.

Summary:

| | Dataset 1: Bankruptcy | Dataset 2: Credit card |
|---|---|---|
| Feature pairs with correlation above 0.9 | 36 of 4,465 | 7 of 253 |
| Features with strong skew | 82 of 95 | 13 of 23 |
| Baseline F1 (predict every row as failed) | 0.062 | 0.362 |
| Train F1 | 0.064 | 0.391 |
| Test F1 | 0.062 | 0.382 |
| Test precision | 0.032 | 0.25 |
| Test recall | 0.955 | 0.818 |
| Test rows with an extreme probability | 0.999 | 0.086 |
| Test F1 in Week 9 (gradient boosting) | 0.349 | 0.478 |

Gaussian Naive Bayes runs on both datasets, but its assumptions are partly broken. For bankruptcy it does no better than predicting every row as failed, and it is extremely overconfident. For credit card it only slightly beats the baseline. It is worse than the Week 9 model for both. The main limits are one split, default settings and no threshold tuning, and only 44 test failures for bankruptcy.